## Webpage Extraction and Embedding (PolyU CUS)

### 1. Extracting raw text data

In [9]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings

from chromadb.config import Settings
from chromadb import Client, PersistentClient

from concurrent.futures import ThreadPoolExecutor
import re

In [10]:
cus_URL = "https://www.polyu.edu.hk/cus/"
start_idx, stop_idx = 6450, -850

docs = []
unwanted_metadata = ["language"]

def bs4_regex_enhance(html: str):
    soup = BeautifulSoup(html, "lxml")                  # Strip the html syntax
    text = re.sub(r"\n\n+", "\n\n", soup.text).strip()  # Strip the excess whitespaces
    return text

cus_loader = RecursiveUrlLoader(
    url=cus_URL,
    base_url=cus_URL,
    prevent_outside=True,
    exclude_dirs=[
        cus_URL+"about-ous",
        cus_URL+"Sitemap", 
        cus_URL+"Search-Result", 
        cus_URL+"Staff",
        cus_URL+"internal",
        cus_URL+"nationaleducation",
        cus_URL+"Undergraduate-Studies-Support/Staff",
    ],
    extractor=bs4_regex_enhance
)

docs_lazy = cus_loader.lazy_load()
for doc in docs_lazy:
    #print(doc)
    doc.page_content = doc.page_content[start_idx:stop_idx]
    for key in unwanted_metadata:
        del doc.metadata[key]
    docs.append(doc)

In [11]:
print(f"Extracted number of webpages in CUS: {len(docs)}")
print(docs[15])

'''
idx = 5
print(docs[idx].metadata.get('source'))
print(docs[idx].page_content)
#print(docs[idx].page_content[1300:-300])
'''

Extracted number of webpages in CUS: 41
page_content='						Student
												

													4-Year Undergraduate Student
												

													Discipline-Specific Requirements (DSR) / Major
												

Discipline-Specific Requirements (DSR) / Major

DSR/Major subjects

DSR/Major subjects are designed aiming at developing students' fundamental discipline-specific knowledge and the skills they need to function effectively as a beginning professional in their chosen field. Particular emphasis is given to the development of students’ generic competencies in the professional context within the discipline-specific curriculum. 

 

 

' metadata={'source': 'https://www.polyu.edu.hk/cus/Student/4-Year-Undergraduate-Student/Discipline-Specific-Requirements-Major', 'content_type': 'text/html; charset=utf-8', 'title': 'Discipline-Specific Requirements (DSR) / Major | College of Undergraduate Studies'}


"\nidx = 5\nprint(docs[idx].metadata.get('source'))\nprint(docs[idx].page_content)\n#print(docs[idx].page_content[1300:-300])\n"

### 2. Text Splitting

In [12]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(docs)

for i, chunk in enumerate(chunks):
    source = chunk.metadata.get("source", "N/A")
    url_end = re.search(r'/([^/]+)/?$', source).group(1)
    chunk.metadata["chunk_id"] = f"PolyU_CUS_{url_end}_chunk_{i}"

In [13]:
print(chunks[20])

page_content='Depending on the destination, duration and nature, the cost of each non-local CAR and SL subject can vary significantly. The final actual expenses for each student, even in the same subject, will most likely differ from the estimated expenses due to various reasons, such as fluctuation in airfare, visa requirement for different students, actual travel medicine expenses, and unexpected incidents during the trip etc. The amount of OAF for each subject will be announced on eStudent (under subject search) before the subject registration period. The unspent balance of the OAF, if any, will be refunded to students after completing the subject.
What happens if I take a second non-local CAR or SL subject?' metadata={'source': 'https://www.polyu.edu.hk/cus/Non-local-GUR-Study/Non-local-Study-Fund/Non-local-Study-Fund', 'content_type': 'text/html; charset=utf-8', 'title': 'Non-local Study Fund | College of Undergraduate Studies', 'description': 'At Hong Kong Polytechnic University,

### 3. Document Embedding in Chroma

In [14]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "polyu_cus_webpage" if not SINGLE else "vaa_documents"

In [15]:
embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

client = Client(Settings())
client = PersistentClient(path="../chroma_db")
collection = client.get_collection(name=collection_name)

In [16]:
def generate_embedding(chunk):
    return embedding_function.embed_query(chunk.page_content)
with ThreadPoolExecutor() as executor:
    embeddings = list(executor.map(generate_embedding, chunks))

for i, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk.page_content], 
        metadatas=[chunk.metadata], 
        embeddings=[embeddings[i]],
        ids=[str(i + 0)]
    )

vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

print(f"Added {len(chunks)} chunks into ChromaDB to {collection_name}")

Added 106 chunks into ChromaDB to vaa_documents


/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_57545/1224759979.py:14: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(


### 4. Simple Testing

In [17]:
query = "What is the general university requirement for undergraduate student?"
results = vectorStore.similarity_search(query, k=5)

for result in results:
    print("========================================================")
    print(f"Content: {result.page_content}...")
    print(f"Source: {result.metadata.get('source')}")
    print(f"Chunk ID: {result.metadata.get('chunk_id')}\n")

Content: dent
												

													4-Year Undergraduate Student
												

													General University Requirements (GUR)
												

General University Requirements (GUR)

GUR for 4-Year Undergraduate Student 

 

Admitted in 2021/22 or before

Freshman Seminar

Language & Communication Requirements

Leadership & Intra-Personal Development

Cluster-Area Requirements

Service-Learning

Healthy Lifestyle

 

Admitted from 2022/23

Artificial Intelligence and Data Analytics Requirement

Innovation and Entrepreneurship Requirement

Language & Communication Requirements

Leadership Education and Development

Cluster-Area Requirements

Service-Learning

Healthy Lifestyle

 

For the details of curriculum framework of the General University Requirements (GUR), please click here....
Source: https://www.polyu.edu.hk/cus/Student/4-Year-Undergraduate-Student/General-University-Requirements
Chunk ID: PolyU_CUS_General-University-Requirements_chunk_40

Content: Minimum credit requir